### Notebook 范围

### 目的
比较 O3、EP100 W1+W2、U60N10、U60N50 的 ensemble spread onset，并同时输出绝对日历时间和相对初始化 lead day 两种视角。

### 科学问题
动力变量 spread 是否早于 O3 spread？这种“早”是在日历季节中更早，还是仅仅相对于不同初始化日期更早？

### 方法
spread(t)=成员标准差；standardized spread=(spread(t)-spread(t0))/std；5-day running mean 首次超过自身最大值的 50% 且至少维持 5 天定义为 onset。EP100 W1+W2 使用 `mean(-ep2)`，100 hPa，40-80N，代表向上传播的 W1+W2 wave activity，不是 EP flux divergence。

### 预期输出
`outputs/figures/04_spread_timing/` 下输出两张主图：一张以 lead day 为 x 轴，一张以 month-day 为 x 轴；`outputs/tables/04_spread_timing/04_spread_onset_timing_summary_all_cases_v3.csv` 保存完整 onset 表。

### 解读
lead-day 图回答“初始化后多少天出现 spread”；month-day 图回答“实际季节中何时出现 spread”。如果 EP/U onset 在 lead-day 和 month-day 两个视角中都早于 O3，则支持 dynamics-first uncertainty pathway。

### 注意
旧版 helper 对只有 lead-day 坐标的 EPFlux 会把 `onset_doy` 写成 `lead_day+1`，因此可能出现看似模拟前已经 spread 的假象。本 notebook 显式用 `case_time_doy(case, date)` 修正绝对日期。


### 导入与公共路径

### 目的
加载 Extention_analysis 的公共函数、数据路径、输出目录和绘图参数。

### 科学问题
保证 O3、EP100 和 U60N 的 spread onset 使用同一套 cleaned hindcast 产品。

### 方法
从 `hindcast_extension_utils.py` 读取 case 发现、O3/EPFlux/U 加载、日期转换、weighted mean、保存图和 figure manifest 工具。

### 预期输出
打印工作目录和 Hindcast 数据目录，不产生图。

### 解读
路径正确说明后续计算只读取 `/mnt/soclim0/public_data/weiji/Hindcast`，输出只写入 `code_cleaned/Hindcast_experiment/Extention_analysis/outputs`。

### 注意
Notebook 不修改原始数据。


In [ ]:
from pathlib import Path
import os
import sys

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib-code-cleaned")

WORK_ROOT = Path("/home/weiji/restart_exam/code_cleaned/Hindcast_experiment/Extention_analysis")
if str(WORK_ROOT) not in sys.path:
    sys.path.insert(0, str(WORK_ROOT))

import math
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.ticker import FixedLocator

import hindcast_extension_utils as hx
from hindcast_extension_utils import *

for d in [FIG_DIR, TAB_DIR, CACHE_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("WORK_ROOT", WORK_ROOT)
print("HINDCAST_ROOT", HINDCAST_ROOT)


### 计算 spread onset 表

### 目的
对每个 case 和每个变量计算 spread onset，并同时记录相对初始化 lead day 与绝对 calendar day/month-day。

### 科学问题
EP100 W1+W2、U60N10、U60N50 的成员分歧是否早于 O3 partial column 的成员分歧？之前 EPFlux DOY 偏早是否只是 lead-day 坐标误读？

### 方法
O3 使用 `load_hindcast_o3`；EP100 W1+W2 使用 wave1 与 wave2 的 `ep2` 相加后取 `-ep2`，在 100 hPa、40-80N 做 cos(lat) 加权平均；U 使用 `compute_u60` 在 10 hPa 和 50 hPa。onset_index 来自 `compute_spread_onset`，绝对日历时间使用 `case_time_doy(case, date)[onset_index]` 重新计算。

### 预期输出
`04_spread_onset_timing_summary_all_cases_v3.csv`，包含 `onset_lead_day`、`onset_calendar_doy`、`onset_month_day` 和 `raw_onset_doy_from_helper`。

### 解读
如果 `raw_onset_doy_from_helper` 对 EPFlux 很小，但 `onset_calendar_doy` 合理落在初始化后，说明旧图的异常来自日期坐标解释，而不是物理 spread 出现在模拟前。

### 注意
calendar day 只按非闰年 Jan-May 解释；hindcast 这里的日历窗口本身也是 Jan-May 诊断，不涉及真实公历年份闰日。


In [ ]:
VERSION_TAG = "v3"
fig_dir = figure_dir("04_spread_timing")
tab_dir = table_dir("04_spread_timing")

VARIABLE_ORDER = ["EP100_W1W2", "U60N10", "U60N50", "O3"]
MARKERS = {"EP100_W1W2": "^", "U60N10": "o", "U60N50": "s", "O3": "D"}
COLORS = {"EP100_W1W2": "tab:purple", "U60N10": "tab:orange", "U60N50": "tab:green", "O3": "tab:blue"}
LABELS = {
    "EP100_W1W2": "EP100 W1+W2 spread",
    "U60N10": "U60N10 spread",
    "U60N50": "U60N50 spread",
    "O3": "O3 spread",
}


def _mmdd_label_from_doy(doy):
    if not np.isfinite(doy):
        return ""
    month, day = doy_to_mmdd(int(round(float(doy))))
    return f"{month:02d}-{day:02d}"


def _absolute_onset_from_case_date(case, date, onset_index):
    if not np.isfinite(onset_index):
        return np.nan, ""
    idx = int(onset_index)
    doys = case_time_doy(case, np.asarray(date))
    if idx < 0 or idx >= len(doys):
        return np.nan, ""
    doy = float(doys[idx])
    return doy, _mmdd_label_from_doy(doy)


def _append_onset(rows, case, variable, da, date):
    if da is None:
        log_message(f"{case}: missing {variable} for spread timing")
        return
    onset = compute_spread_onset(da)
    onset_index = onset.get("onset_index", np.nan)
    onset_calendar_doy, onset_month_day = _absolute_onset_from_case_date(case, date, onset_index)
    meta = parse_case_name(case)
    rows.append({
        "case": case,
        "year": meta["year"],
        "init_month": meta["init_month"],
        "config": meta["config"],
        "perturbation": meta["perturbation"],
        "variable": variable,
        "onset_index": onset_index,
        "onset_lead_day": onset.get("onset_lead_day", onset_index),
        "onset_calendar_doy": onset_calendar_doy,
        "onset_month_day": onset_month_day,
        "raw_onset_doy_from_helper": onset.get("onset_doy", np.nan),
        "threshold": onset.get("threshold", np.nan),
        "n_time": int(da.sizes.get("lead_time", da.sizes.get("time", 0))),
        "definition": "5-day running standardized ensemble spread > 50% of its own maximum for 5 days",
    })


inv = discover_hindcast_cases()
rows = []
for case in inv.loc[inv["has_partial_O3"], "case"]:
    o3, o3_date = load_hindcast_o3(case)
    _append_onset(rows, case, "O3", o3, o3_date if o3_date is not None else np.arange(o3.sizes["lead_time"]) if o3 is not None else [])

    ep1, ep1_date = load_epflux(case, "wave1")
    ep2, ep2_date = load_epflux(case, "wave2")
    if ep1 is not None and ep2 is not None:
        ep = coslat_weighted_mean(
            -(ep1.sel(plev=10000, method="nearest") + ep2.sel(plev=10000, method="nearest")),
            LAT_EP,
        )
        ep = ep.assign_coords(date=("lead_time", np.asarray(ep1_date)[: ep.sizes["lead_time"]]))
        _append_onset(rows, case, "EP100_W1W2", ep, ep1_date)
    else:
        log_message(f"{case}: missing EP wave1 or wave2 for spread timing")

    for plev in [10, 50]:
        u, u_date = compute_u60(case, plev)
        _append_onset(rows, case, f"U60N{plev}", u, u_date if u_date is not None else np.arange(u.sizes["lead_time"]) if u is not None else [])

spread = pd.DataFrame(rows)
spread["variable"] = pd.Categorical(spread["variable"], categories=VARIABLE_ORDER, ordered=True)
spread = spread.sort_values(["year", "init_month", "case", "variable"]).reset_index(drop=True)
spread_csv = tab_dir / f"04_spread_onset_timing_summary_all_cases_{VERSION_TAG}.csv"
spread.to_csv(spread_csv, index=False)
# Keep the historical root-level filename available for downstream notebooks.
spread.to_csv(TAB_DIR / "04_spread_onset_timing_summary_all_cases.csv", index=False)
print(spread[["case", "variable", "onset_lead_day", "onset_calendar_doy", "onset_month_day", "raw_onset_doy_from_helper"]].to_string(index=False))


### 绘制相对初始化 lead-day spread onset 图

### 目的
使用距离初始化的 lead day 比较各变量 spread onset，避免不同初始化月份造成的绝对日期偏移。

### 科学问题
在每个 hindcast 自己的初始化之后，EP100/U 的成员分歧是否比 O3 更早出现？

### 方法
x 轴为 `onset_lead_day`；y 轴为 case；marker 表示变量。图标题中写明 spread 定义：ensemble std，经 5-day running 标准化后超过自身最大值 50% 并维持 5 天。

### 预期输出
`04_spread_onset_timing_summary_lead_day_v3.png/pdf`。

### 解读
同一行中，紫色/橙色/绿色 marker 位于蓝色 O3 marker 左侧，说明动力变量 spread 领先 O3 spread。

### 注意
lead-day 图适合比较初始化后的相对演变，但不能说明这些事件在真实季节中发生于哪个月日。


In [ ]:
def _ordered_cases(df):
    if df.empty:
        return []
    tmp = df[["case", "year", "init_month"]].drop_duplicates().copy()
    tmp["year_i"] = tmp["year"].astype(int)
    tmp["init_i"] = tmp["init_month"].astype(int)
    return tmp.sort_values(["year_i", "init_i", "case"])["case"].tolist()


def _plot_spread_summary(df, x_col, xlabel, title, name, calendar_axis=False):
    cases = _ordered_cases(df)
    fig, ax = plt.subplots(figsize=(11.8, max(5.2, 0.38 * len(cases) + 2.0)), constrained_layout=True)
    if df.empty or not cases:
        ax.axis("off")
        ax.text(0.5, 0.5, "No spread onset diagnostics", ha="center", va="center")
    else:
        ymap = {c: i for i, c in enumerate(cases)}
        for var in VARIABLE_ORDER:
            sub = df[(df["variable"] == var) & np.isfinite(df[x_col])]
            if sub.empty:
                continue
            ax.scatter(
                sub[x_col],
                [ymap[c] for c in sub["case"]],
                marker=MARKERS.get(var, "o"),
                s=88,
                color=COLORS.get(var),
                label=LABELS.get(var, var),
                edgecolor="black",
                linewidth=0.7,
                zorder=3,
            )
        ax.set_yticks(range(len(cases)), cases)
        ax.set_xlabel(xlabel)
        ax.set_title(title + "\nspread(t)=ensemble std; onset=5-day running standardized spread > 50% max for 5 days", fontsize=12)
        ax.legend(ncol=2, fontsize=9, title="Variable")
        ax.grid(True, axis="x", alpha=0.25)
        if calendar_axis:
            ticks = [1, 32, 60, 91, 121, 152]
            labels = ["Jan 1", "Feb 1", "Mar 1", "Apr 1", "May 1", "Jun 1"]
            ax.set_xlim(1, 152)
            ax.xaxis.set_major_locator(FixedLocator(ticks))
            ax.set_xticklabels(labels)
        else:
            xmax = np.nanmax(df[x_col].values) if np.isfinite(df[x_col]).any() else 1
            ax.set_xlim(left=0, right=max(35, math.ceil(xmax / 10) * 10 + 5))
    savefig(
        fig,
        name,
        fig_dir=fig_dir,
        notebook="04_spread_timing_multicase.ipynb",
        scientific_question="EP/vortex spread 是否早于 O3 spread？",
        variables_windows="O3 partial column; EP100 W1+W2=mean(-ep2) at 100 hPa over 40-80N; U60N10/U60N50; onset definition in title.",
        interpretation="EP/U marker 在 O3 左侧支持 dynamics-first uncertainty propagation。",
        caveat="阈值 onset 是经验诊断；lead-day 图不表示绝对季节日期。",
        csv_table=spread_csv,
    )
    plt.show()

_plot_spread_summary(
    spread,
    x_col="onset_lead_day",
    xlabel="Spread onset lead day from initialization",
    title="Spread onset timing relative to initialization",
    name=f"04_spread_onset_timing_summary_lead_day_{VERSION_TAG}",
    calendar_axis=False,
)


### 绘制绝对日历 month-day spread onset 图

### 目的
把同一批 spread onset 映射回真实季节中的 month-day，直接检查 EP100 W1+W2 spread 是否真的发生在初始化之前。

### 科学问题
不同初始化月份的 hindcast 中，动力 spread 和 O3 spread 在日历季节上的先后顺序是什么？EPFlux 旧图中“DOY 不到 20”的现象是否只是 lead-day 坐标误读？

### 方法
x 轴为 `onset_calendar_doy`，由 `case_time_doy(case, date)[onset_index]` 计算。对于 EPFlux 这类 date=0,1,2... 的产品，`case_time_doy` 会自动加上 case 初始化月的 DOY 偏移，例如 0008-02 的 lead day 0 对应 Feb 1。

### 预期输出
`04_spread_onset_timing_summary_calendar_month_day_v3.png/pdf`。

### 解读
如果 EP100 marker 落在对应 case 初始化月之后，就说明模拟前 spread 的直觉是旧坐标误读；如果 EP/U 在 month-day 视角仍早于 O3，说明动力 spread 在实际季节中也先发生。

### 注意
绝对日期图适合看季节先后，但不同初始化月份的可用 hindcast 时间长度不同，later-initialized cases 不能用于判断初始化前的 precursor。


In [ ]:
_plot_spread_summary(
    spread,
    x_col="onset_calendar_doy",
    xlabel="Spread onset calendar date",
    title="Spread onset timing in calendar month-day",
    name=f"04_spread_onset_timing_summary_calendar_month_day_{VERSION_TAG}",
    calendar_axis=True,
)
write_figure_guide()
